# Deep Learning in Audio Applications
### Teach machines to listen: your own audio-to-text notebook

You will take speech, turn it into text, see where that breaks, build the two ideas that let a computer read a sequence
(an **RNN**, then an **LSTM**), and end with **meeting minutes** from a conversation with two speakers.

Run the cells from top to bottom. Cells marked **YOUR TURN** have one line for you to write; a check tells you straight away if it is right.

| | |
|---|---|
| 1 | Audio to text with Python |
| 2 | Where transcription struggles |
| 3 | RNNs: reading audio as a sequence |
| 4 | LSTMs: fixing the short memory |
| 5 | Meeting minutes |

## 0. Setup (run once, about a minute)
On Colab: **Runtime > Change runtime type > T4 GPU** makes Whisper several times faster (the free CPU works too, just slower).
Upload `workshop_files.zip` from the facilitators to the Files panel on the left before running this cell.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")                  # avoids a library clash on some Windows installs

if importlib.util.find_spec("whisper") is None:                       # Whisper: speech-to-text from OpenAI
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("workshop_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("workshop_files.zip").extractall(".")              # the code and audio for this workshop
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)   # ../prototype when run from the repo
PROTO or sys.exit("Upload workshop_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg (reads audio files):", "found" if shutil.which("ffmpeg") else "NOT FOUND - install it (Colab has it already)")
print("ready")

## Live hook demo (Agenda item 1 -- the facilitator runs this live at the start of the session)
A mock CCA meeting (`demo/script.txt`, two speakers) is acted out or played, the room calls out
the decisions and to-dos, then **one button** shows the app producing the same list as minutes.
The full live-recording version (microphone, Colab) is in the facilitator-only notebook
`demo/hook_demo.ipynb`; this cell just runs the same code on the committed script text, so you can
see what it does without recording anything.


In [ ]:
from demo import run_demo   # prototype/demo.py; PROTO is already on sys.path (cell above)
_ = run_demo()   # no args: runs the committed demo/script.txt and prints the minutes


## 1. Audio to text with Python
Whisper is a ready-made speech-to-text model. Three lines: load it, give it a file, read the text.
Bigger models hear more but run slower: `tiny.en`, `base.en`, `small.en`, `medium.en` (the `.en` ones know only English). We pick a size that fits your machine.

In [ ]:
import torch, whisper

MODEL = "small.en" if torch.cuda.is_available() else "base.en"        # a GPU can afford the bigger model
print("using", MODEL, "on", "the GPU" if torch.cuda.is_available() else "the CPU")
model = whisper.load_model(MODEL)
result = model.transcribe("audio/sample_meeting.wav", language="en")
print(result["text"])

Our pipeline wants more than one block of text: it wants **who said what**. `get_transcript` listens for pauses and
starts a new turn each time one person stops and the next starts (here: a pause longer than 1.5 s).

In [ ]:
os.environ["WHISPER_MODEL"] = MODEL          # the pipeline's transcriber reads this
from transcriber import get_transcript

transcript = get_transcript("audio/sample_meeting.wav", pause_gap=1.5)
print(transcript)

## 2. Where transcription struggles
Real recordings are not clean. The three things that hurt most: **background noise**, **a poor microphone**, and **accents**.
We score by how many of the words Whisper gets right (`word_accuracy`: 1.0 = every word).

In [ ]:
import numpy as np
from audio_check import word_accuracy

SAID = ("Venue is confirmed for the 14th, catering still pending. "
        "Can you follow up with catering by Friday? Yes, I'll email them today.")
clean = whisper.load_audio("audio/sample_short.wav")          # the same three sentences, 14 seconds

def hear(samples):
    return model.transcribe(samples, language="en")["text"].strip()

print("clean      ", f"{word_accuracy(SAID, hear(clean)):.0%}", "|", hear(clean))

**Noise.** Add random hiss to the recording. How loud can it get before Whisper breaks? (try 0.02, 0.05, 0.1, 0.2)

In [ ]:
NOISE = 0.1
noisy = (clean + NOISE * np.random.default_rng(0).normal(size=clean.shape)).astype("float32")
heard = hear(noisy)
print(f"noise {NOISE}:", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**A poor microphone.** Keep only every 4th sample, like a cheap mic or a phone line. The high sounds (s, t, f) disappear.

In [ ]:
poor = np.repeat(clean[::4], 4).astype("float32")
heard = hear(poor)
print("poor mic  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**Accents.** Record yourself on your phone reading the same three sentences, with about 2 seconds between them:

> Venue is confirmed for the 14th, catering still pending. / Can you follow up with catering by Friday? / Yes, I'll email them today.

Upload it to the Files panel, put its name below, and compare with the clean voice. What did Whisper get wrong, and why?

In [ ]:
MY_VOICE = None            # for example "my_voice.m4a"
if MY_VOICE:
    mine = whisper.load_audio(MY_VOICE)
    heard = hear(mine)
    print("my voice  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)
else:
    print("Set MY_VOICE to a file you uploaded, then run this cell again.")

## 3. RNNs: reading audio as a sequence
Sound is a long list of numbers. A computer cannot read all of it at once, so we cut it into frames and read **one frame at a time**,
carrying a **memory** from frame to frame. That is an RNN. First, what a frame is:

In [ ]:
from audio_features import mfcc

samples = whisper.load_audio("audio/sample_meeting.wav")
frames = mfcc(samples)
print(f"{len(samples) / 16000:.0f} seconds of audio  ->  {frames.shape[0]} frames x {frames.shape[1]} numbers (one frame every 10 ms)")

### YOUR TURN (Part A): one step of an RNN
At every step the RNN mixes **the new input** with **its old memory**:

`new memory = tanh( W_x · x  +  W_h · old memory  +  b )`

In numpy `·` is `@`, and tanh is `np.tanh`. Write that one line.

In [ ]:
import numpy as np, hands_on

def rnn_step(x, h_prev, W_x, W_h, b):
    # TODO: return the new memory (the line above, in numpy)
    return None

hands_on.rnn_step = rnn_step
hands_on.MEMORY_ON = True
hands_on.part_a()

Now **break the memory**: set `hands_on.MEMORY_ON = False` in the cell above (no old memory in the formula) and run it again. What happens to the first word?

## 4. LSTMs: fixing the short memory
An RNN's memory fades a little at every step, so words from long ago are lost. An **LSTM** keeps a separate notebook and
decides at every step how much to erase (the **forget gate**) and how much to write (the **input gate**).
Our model is an *hourglass* of 5 LSTM layers: in the middle layer each step covers 4 words, so memory has far fewer steps to survive.

In [ ]:
from lstm_model import fold_time, unfold_time
x = np.arange(16.0).reshape(8, 2)
print("8 steps of 2 numbers, folded:", fold_time(x).shape, "(4 steps of 4); unfolded back is identical:", np.array_equal(unfold_time(fold_time(x)), x))

### YOUR TURN (Part B): the LSTM notebook update
`new notebook = (how much to keep) × (old notebook) + (how much to add) × (the new note)`

`f` is the eraser, `i` is the pencil, `g` is the new note, `c_prev` is the old notebook. Write that one line.

In [ ]:
def lstm_cell_update(f, i, g, c_prev):
    # TODO: return the new notebook (the line above, in numpy)
    return None

hands_on.lstm_cell_update = lstm_cell_update
hands_on.FORGET_KNOB = 0          # try -6, -3, 0, 3, 6: negative erases more, positive keeps more
hands_on.part_b()

Turn the knob to **-6** and **+6** and run the cell again. What happens to the first word, and to the trained model's score? Why do both extremes hurt it?

## 5. Meeting minutes
Now put it together: your two-speaker transcript goes through the trained LSTM, which labels every turn as a
**decision**, an **action item** or plain **discussion**, and the minutes are written from the labels.

In [ ]:
from lstm_model import run_pipeline
from meeting_minutes import label_turns_model, generate_minutes

turns, output, turn_ends = run_pipeline(transcript=transcript)
labels = label_turns_model(turns, output, turn_ends)
for (speaker, text), label in zip(turns, labels):
    print(f"{label['category']:<12} {speaker}: {text}")
print()
print(generate_minutes(turns, labels))

**Look closely.** Did it find both decisions? (It misses decisions stated as plain facts, such as "Venue is confirmed", and
a bare "Sounds good." after a proposal. That is where a model trained on few examples is weakest.)

### Your own meeting
Type a short meeting of your own (one line per turn, `Name: what they said`) and see what the model makes of it.

In [ ]:
MY_MEETING = '''
Sam: Should we move the meeting to Friday?
Lee: Sounds good.
Sam: Lee, can you book the room by Thursday?
Lee: Sure.
'''
turns, output, turn_ends = run_pipeline(transcript=MY_MEETING.strip())
labels = label_turns_model(turns, output, turn_ends)
for (speaker, text), label in zip(turns, labels):
    print(f"{label['category']:<12} {speaker}: {text}")
print()
print(generate_minutes(turns, labels))

### Going further
* Change `MODEL` in section 1 to `small.en` or `medium.en`. How much does the transcript improve, and how much slower is it?
* Record a real two-person conversation (pause between speakers) and run it through `get_transcript`.
* Section 2 again with your own noise level or a different recording.